<a href="https://colab.research.google.com/github/kumisganteng/Data-Analytics-Projects/blob/energy_data_cleaning/Energy_Consumption_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Data Ingestion

##1.1 Data Loading

In [1]:
import pandas as pd

AEP_hourly_energy_dataset = "AEP_hourly.csv"
energy_data = pd.read_csv(AEP_hourly_energy_dataset)

##1.2 Initial Data Inspection

In [2]:
energy_data.shape

(121273, 2)

In [3]:
energy_data.head()

,Datetime,AEP_MW
0,2004-12-31 01:00:00,13478.0
1,2004-12-31 02:00:00,12865.0
2,2004-12-31 03:00:00,12577.0
3,2004-12-31 04:00:00,12517.0
4,2004-12-31 05:00:00,12670.0


In [4]:
energy_data.tail()

,Datetime,AEP_MW
121268,2018-01-01 20:00:00,21089.0
121269,2018-01-01 21:00:00,20999.0
121270,2018-01-01 22:00:00,20820.0
121271,2018-01-01 23:00:00,20415.0
121272,2018-01-02 00:00:00,19993.0


In [5]:
energy_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121273 entries, 0 to 121272
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   Datetime  121273 non-null  object 
 1   AEP_MW    121273 non-null  float64
dtypes: float64(1), object(1)
memory usage: 1.9+ MB


In [6]:
energy_data.describe()

,AEP_MW
count,121273.000000
mean,15499.513717
std,2591.399065
min,9581.000000
25%,13630.000000
50%,15310.000000
75%,17200.000000
max,25695.000000


In [7]:
energy_data.isnull().sum()

,0
Datetime,0
AEP_MW,0


In [8]:
energy_data.duplicated().sum()

np.int64(0)

##1.3 Datetime Range Check

In [9]:
# Check Datetime Range
print(energy_data['Datetime'].min())
print(energy_data['Datetime'].max())

# Check Missing Hours
energy_data['Datetime'] = pd.to_datetime(energy_data['Datetime'])
expected = pd.date_range(start=energy_data['Datetime'].min(),
                         end=energy_data['Datetime'].max(),
                         freq='h')
missing_hours = expected.difference(energy_data['Datetime'])

# Checking
print(f"Missing hours: {len(missing_hours)}")
print(f"Duplikasi: {energy_data.duplicated().sum()}")
print(f"Missing Values: {energy_data.isnull().sum()}")

2004-10-01 01:00:00
2018-08-03 00:00:00
Missing hours: 27
Duplikasi: 0
Missing Values: Datetime    0
AEP_MW      0
dtype: int64


#2. Data Cleaning

##2.1 Handle Missing Hours

###Set Datetime As Index

In [10]:
energy_data = energy_data.set_index('Datetime')

In [ ]:
print(f"Baris setelah set index: {len(energy_data)}")

In [11]:
duplikat_index = energy_data.index.duplicated().sum()
print(f"Duplikasi pada index: {duplikat_index}")

Duplikasi pada index: 4


In [12]:
energy_data = energy_data[~energy_data.index.duplicated(keep='first')]

In [13]:
len(energy_data)

121269

###Reindex to Full Hourly Range

In [14]:
full_range = pd.date_range(
    start=energy_data.index.min(),
    end=energy_data.index.max(),
    freq='h'
)
energy_data = energy_data.reindex(full_range)
energy_data.index.name = 'Datetime'

###Checking After Reindex

In [15]:
print(f"\nExpected hours : {len(full_range)}")
print(f"Actual rows    : {len(energy_data)}")
print(f"Selisih        : {len(full_range) - len(energy_data)}")
print(f"Missing values : {energy_data.isnull().sum().values[0]}")



Expected hours : 121296
Actual rows    : 121296
Selisih        : 0
Missing values : 27


###Handle Missing Values with Linear Interpolation

In [16]:
energy_data['AEP_MW'] = energy_data['AEP_MW'].interpolate(method='linear')
#Checking
print(f"\nMissing values after interpolation: {energy_data.isnull().sum().values[0]}")
print(f"Total rows after cleaning: {len(energy_data)}")


Missing values after interpolation: 0
Total rows after cleaning: 121296


##2.2 Data Cleaning Summary


In [18]:
print("=" * 45)
print("         DATA CLEANING SUMMARY")
print("=" * 45)
print(f"Total rows after cleaning  : {len(energy_data)}")
print(f"Missing values             : {energy_data.isnull().sum().values[0]}")
print(f"Duplicate index            : {energy_data.index.duplicated().sum()}")
print(f"Datetime min               : {energy_data.index.min()}")
print(f"Datetime max               : {energy_data.index.max()}")
print(f"AEP_MW min                 : {energy_data['AEP_MW'].min()}")
print(f"AEP_MW max                 : {energy_data['AEP_MW'].max()}")
print(f"AEP_MW mean                : {energy_data['AEP_MW'].mean():.2f}")
print(f"Data type of AEP_MW         : {energy_data['AEP_MW'].dtype}")
print(f"Index type                 : {type(energy_data.index).__name__}")
print("=" * 45)

         DATA CLEANING SUMMARY
Total rows after cleaning  : 121296
Missing values             : 0
Duplicate index            : 0
Datetime min               : 2004-10-01 01:00:00
Datetime max               : 2018-08-03 00:00:00
AEP_MW min                 : 9581.0
AEP_MW max                 : 25695.0
AEP_MW mean                : 15499.15
Data type of AEP_MW         : float64
Index type                 : DatetimeIndex
